# RGCA Real Generation Milestone - Complete Kaggle Notebook

This notebook runs the real-generation milestone from the private Kaggle input dataset. It does **not** use gcloud, does **not** redownload MIMIC, and refuses to silently fall back to mock/stress generation when real generation is intended.

Required Kaggle input: `rgca-real-generation-input`.


## Run Order

Run every cell from top to bottom. This notebook contains everything needed for the real-generation milestone after attaching the private Kaggle dataset `rgca-real-generation-input`.

What this notebook does:

1. Defines paths and helper functions.
2. Sets the real-generation config.
3. Clones or updates `pidoxy/RGCA` from GitHub.
4. Finds and copies the attached private input dataset.
5. Runs a cheap preflight check before model inference.
6. Installs VLM dependencies.
7. Runs real report generation with `hf_vlm`.
8. Writes evaluation summaries and packages the evidence zip.

Important: the preflight cell should pass before the VLM dependency/model generation cells run. If preflight fails, stop there and fix the setup first.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

PROJECT_ROOT = Path('/kaggle/working/RGCA')
WORKING_ROOT = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')

SUBSET_PATH = WORKING_ROOT / 'rgca_hydrated_subset' / 'mimic_subset.jsonl'
RETRIEVAL_RESULTS = WORKING_ROOT / 'rgca_experiments' / 'biomedclip_retrieval_validation_v0' / 'retrieval_results.jsonl'
MISMATCH_RESULTS = WORKING_ROOT / 'rgca_experiments' / 'biomedclip_retrieval_validation_v0' / 'mismatch_results.jsonl'
OUTPUT_DIR = WORKING_ROOT / 'rgca_experiments' / 'real_generation_v0'
EVIDENCE_ZIP = WORKING_ROOT / 'rgca_real_generation_evidence_v0.zip'

def run_command(command, cwd=PROJECT_ROOT):
    print('+', ' '.join(str(part) for part in command))
    return subprocess.run(command, cwd=str(cwd), check=True)

def read_json(path):
    return json.loads(Path(path).read_text())

def read_jsonl(path, limit=None):
    rows = []
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
                if limit is not None and len(rows) >= limit:
                    break
    return rows

def find_input_bundle_root():
    """Find the attached Kaggle dataset even if Kaggle nests it one or two folders deep."""
    if not INPUT_ROOT.exists():
        return None

    subset_matches = list(INPUT_ROOT.rglob('mimic_subset.jsonl'))
    for subset in subset_matches:
        # Expected shape: <root>/rgca_hydrated_subset/mimic_subset.jsonl
        if subset.parent.name == 'rgca_hydrated_subset':
            candidate = subset.parent.parent
            retrieval = candidate / 'rgca_experiments' / 'biomedclip_retrieval_validation_v0' / 'retrieval_results.jsonl'
            mismatch = candidate / 'rgca_experiments' / 'biomedclip_retrieval_validation_v0' / 'mismatch_results.jsonl'
            if retrieval.exists() and mismatch.exists():
                return candidate

    # Fallback for very flat dataset layouts.
    for candidate in INPUT_ROOT.rglob('*'):
        if not candidate.is_dir():
            continue
        if (candidate / 'rgca_hydrated_subset').exists() and (candidate / 'rgca_experiments').exists():
            return candidate
    return None

print('Python:', sys.version)
print('Project root exists:', PROJECT_ROOT.exists())
print('Kaggle input root exists:', INPUT_ROOT.exists())
print('Top-level input dirs:', [str(p) for p in INPUT_ROOT.glob('*')] if INPUT_ROOT.exists() else [])
print('Detected bundle root:', find_input_bundle_root())


In [ ]:
# Real generation configuration
# This notebook is intentionally configured for REAL VLM generation, not the cheap stress/debug backend.

INTENDED_REAL_GENERATION = True
GENERATOR_BACKEND = "hf_vlm"
MODEL_ID = "llava-hf/llava-1.5-7b-hf"
LIMIT = 5

# If the model is gated or Kaggle cannot access it, switch MODEL_ID to another public HF VLM checkpoint.
# Do not change GENERATOR_BACKEND to retrieval_copy_stress for the real generation milestone.
print({
    "INTENDED_REAL_GENERATION": INTENDED_REAL_GENERATION,
    "GENERATOR_BACKEND": GENERATOR_BACKEND,
    "MODEL_ID": MODEL_ID,
    "LIMIT": LIMIT,
})


In [ ]:
# Prepare repo and attached evidence input.
# This cell is safe to rerun. It does not require gcloud or PhysioNet credentials.

if not PROJECT_ROOT.exists():
    run_command(['git', 'clone', 'https://github.com/pidoxy/RGCA.git', str(PROJECT_ROOT)], cwd=WORKING_ROOT)
else:
    print('RGCA repo already exists; pulling latest code:', PROJECT_ROOT)
    run_command(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT)

run_command([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], cwd=PROJECT_ROOT)

input_dir = find_input_bundle_root()
if input_dir is None:
    print('Files visible under /kaggle/input:')
    for path in list(INPUT_ROOT.rglob('*'))[:80] if INPUT_ROOT.exists() else []:
        print(' -', path)
    raise FileNotFoundError(
        'The dataset is attached, but the expected real-generation files were not found. '        'The attached dataset must contain rgca_hydrated_subset/mimic_subset.jsonl and '        'rgca_experiments/biomedclip_retrieval_validation_v0/{retrieval_results.jsonl,mismatch_results.jsonl}.'
    )

print('Using attached input bundle:', input_dir)

for name in ['rgca_hydrated_subset', 'rgca_experiments', 'physionet']:
    src_path = input_dir / name
    dst_path = WORKING_ROOT / name
    if src_path.exists() and not dst_path.exists():
        print('Copying', src_path, '->', dst_path)
        shutil.copytree(src_path, dst_path)
    elif dst_path.exists():
        print('Already prepared:', dst_path)
    else:
        print('Not present in input, skipping:', src_path)

required = {
    'subset': SUBSET_PATH,
    'retrieval_results': RETRIEVAL_RESULTS,
    'mismatch_results': MISMATCH_RESULTS,
}
missing = {key: str(path) for key, path in required.items() if not path.exists()}
if missing:
    raise FileNotFoundError(f'Missing required real-generation inputs after copy: {missing}')

print('Subset rows preview:', len(read_jsonl(SUBSET_PATH, limit=5)), 'preview rows loaded')
print('Retrieval file:', RETRIEVAL_RESULTS)
print('Mismatch file:', MISMATCH_RESULTS)


In [ ]:
# Cheap preflight checks. This cell should pass before any expensive model inference starts.

print('Checking generation runner CLI...')
run_command([sys.executable, 'scripts/run_generation_from_retrieval.py', '--help'], cwd=PROJECT_ROOT)

required = {
    'subset': SUBSET_PATH,
    'retrieval_results': RETRIEVAL_RESULTS,
    'mismatch_results': MISMATCH_RESULTS,
}
missing = {name: str(path) for name, path in required.items() if not path.exists()}
if missing:
    raise FileNotFoundError(f'Preflight failed. Missing required files: {missing}')

subset_preview = read_jsonl(SUBSET_PATH, limit=5)
retrieval_preview = read_jsonl(RETRIEVAL_RESULTS, limit=2)
mismatch_preview = read_jsonl(MISMATCH_RESULTS, limit=2)

if not subset_preview:
    raise RuntimeError('Preflight failed: subset file is empty.')
if not retrieval_preview:
    raise RuntimeError('Preflight failed: retrieval_results.jsonl is empty.')
if not mismatch_preview:
    raise RuntimeError('Preflight failed: mismatch_results.jsonl is empty.')

needed_study_keys = {'study_id', 'image_path', 'report_text', 'split'}
missing_keys = needed_study_keys - set(subset_preview[0])
if missing_keys:
    raise RuntimeError(f'Preflight failed: subset rows are missing keys: {sorted(missing_keys)}')

needed_retrieval_keys = {'target_study', 'retrieved_reports', 'retrieved_studies'}
retrieval_missing = needed_retrieval_keys - set(retrieval_preview[0])
mismatch_missing = needed_retrieval_keys - set(mismatch_preview[0])
if retrieval_missing:
    raise RuntimeError(f'Preflight failed: retrieval rows are missing keys: {sorted(retrieval_missing)}')
if mismatch_missing:
    raise RuntimeError(f'Preflight failed: mismatch rows are missing keys: {sorted(mismatch_missing)}')

print('Subset preview row:')
print(json.dumps(subset_preview[0], indent=2)[:1200])
print('Retrieval preview row:')
print(json.dumps(retrieval_preview[0], indent=2)[:1200])

if GENERATOR_BACKEND == 'hf_vlm':
    try:
        import torch
    except Exception as exc:
        raise RuntimeError('Preflight failed: torch is not importable in this Kaggle runtime.') from exc
    print('CUDA available:', torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError('Preflight failed: enable a Kaggle GPU accelerator before real VLM generation.')
    print('CUDA device:', torch.cuda.get_device_name(0))

print('Preflight passed. It is safe to continue to dependency install and generation.')


In [ ]:
# Install real VLM runtime dependencies.
# Kaggle usually already has torch; this installs the Hugging Face stack needed by hf_vlm.

if GENERATOR_BACKEND == 'hf_vlm':
    run_command([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'transformers>=4.45.0', 'accelerate>=0.30.0', 'pillow', 'sentencepiece'
    ], cwd=PROJECT_ROOT)
else:
    print('Skipping VLM dependency install because GENERATOR_BACKEND is', GENERATOR_BACKEND)


In [ ]:
# Run the real generation milestone.
# This cell removes stale stress/debug outputs first so the displayed evidence cannot come from an old run.

GENERATOR_BACKEND = globals().get('GENERATOR_BACKEND', os.environ.get('RGCA_GENERATOR_BACKEND', 'retrieval_copy_stress'))
MODEL_ID = globals().get('MODEL_ID', os.environ.get('RGCA_VLM_MODEL_ID', ''))
LIMIT = int(globals().get('LIMIT', os.environ.get('RGCA_GENERATION_LIMIT', '20')))
INTENDED_REAL_GENERATION = bool(globals().get('INTENDED_REAL_GENERATION', GENERATOR_BACKEND == 'hf_vlm'))

print('Effective generation config:')
print(json.dumps({
    'intended_real_generation': INTENDED_REAL_GENERATION,
    'generator_backend': GENERATOR_BACKEND,
    'model_id': MODEL_ID,
    'limit': LIMIT,
}, indent=2))

if INTENDED_REAL_GENERATION and GENERATOR_BACKEND != 'hf_vlm':
    raise RuntimeError('Refusing to run: INTENDED_REAL_GENERATION=True but GENERATOR_BACKEND is not hf_vlm.')
if GENERATOR_BACKEND == 'hf_vlm' and not MODEL_ID:
    raise RuntimeError('GENERATOR_BACKEND=hf_vlm requires MODEL_ID, e.g. llava-hf/llava-1.5-7b-hf.')

if OUTPUT_DIR.exists():
    print('Removing stale output directory:', OUTPUT_DIR)
    shutil.rmtree(OUTPUT_DIR)
if EVIDENCE_ZIP.exists():
    print('Removing stale evidence zip:', EVIDENCE_ZIP)
    EVIDENCE_ZIP.unlink()

command = [
    sys.executable, 'scripts/run_generation_from_retrieval.py',
    '--subset', str(SUBSET_PATH),
    '--retrieval-results', str(RETRIEVAL_RESULTS),
    '--mismatch-results', str(MISMATCH_RESULTS),
    '--output-dir', str(OUTPUT_DIR),
    '--generator', GENERATOR_BACKEND,
    '--limit', str(LIMIT),
    '--require-images',
]

if GENERATOR_BACKEND == 'hf_vlm':
    command.extend(['--model-id', MODEL_ID, '--require-real-generator'])

run_command(command)

manifest_path = OUTPUT_DIR / 'generation_manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing generation manifest: {manifest_path}')

manifest = read_json(manifest_path)
print(json.dumps(manifest, indent=2))

if manifest.get('generator_backend') != GENERATOR_BACKEND:
    raise AssertionError(f'Manifest backend mismatch: {manifest.get("generator_backend")} != {GENERATOR_BACKEND}')
if int(manifest.get('eval_size', -1)) != LIMIT:
    raise AssertionError(f'Manifest eval_size mismatch: {manifest.get("eval_size")} != {LIMIT}')
if GENERATOR_BACKEND == 'hf_vlm':
    if not manifest.get('model_id'):
        raise AssertionError('Manifest missing model_id for hf_vlm run.')
    image_validation = manifest.get('image_validation') or {}
    if image_validation.get('missing_image_count') not in (0, None):
        raise AssertionError(f'Image validation found missing images: {image_validation}')

shutil.make_archive(str(EVIDENCE_ZIP.with_suffix('')), 'zip', root_dir=str(OUTPUT_DIR))
print('Evidence zip:', EVIDENCE_ZIP)
print('Evidence zip MB:', round(EVIDENCE_ZIP.stat().st_size / (1024 * 1024), 3))


In [ ]:
# Inspect generated evidence and metrics.

for mode in ['no_retrieval', 'retrieval', 'mismatch']:
    generation_file = OUTPUT_DIR / f'generations_{mode}.jsonl'
    eval_file = OUTPUT_DIR / f'evaluation_{mode}' / 'evaluation_summary.json'
    print(f'\n== {mode} ==')
    print('generation_file:', generation_file, 'exists:', generation_file.exists())
    if generation_file.exists():
        sample = read_jsonl(generation_file, limit=1)
        if sample:
            print(json.dumps(sample[0], indent=2)[:3000])
    if eval_file.exists():
        print('evaluation:')
        print(json.dumps(read_json(eval_file), indent=2))

print('\nDownload this file from the Kaggle Output panel:')
print(EVIDENCE_ZIP)
